# Day 8 ｜ 函数与代码组织

**日期**：2026-10-26 周一　**Day 8 / 84**　**第 2 周第 1 天**

今天补的是**你已经在用的东西的原理**。`week01/day06_cleaner.py` 里已经有 6 个函数了——
`normalize_key`、`to_number`、`to_halfwidth`、`display_width`、`pad_right`、`run_self_checks`。
你昨天用它们的时候没有任何障碍，所以今天学起来会很快。

## 零散时间安排

| 节 | 内容 | 用时 | 优先级 |
|---|---|---|---|
| 复习 | D-1 / D-3 / D-7 | 5分 | 必做 |
| 1 | 为什么要有函数（三个理由） | 5分 | 必做 |
| 2 | def / return / 没有 return 会怎样 | 7分 | 必做 |
| 3 | 参数四件套：位置·默认值·关键字·*args | 10分 | 必做 |
| 4 | 返回多个值（元组解包） | 5分 | 必做 |
| 5 | ⚠️ 作用域与全局变量的坑 | 10分 | **本周最重要** |
| 6 | docstring 与类型标注 | 6分 | 必做（对 AI 有实效）|
| 7 | class 读懂即可 | 8分 | 只读不写 |
| 练习 | A 重构你的清洗器 / B 加类型 / C 读 class | 20分 | **A 必须做** |

---
## 🔁 开场复习（5 分钟）

三个间隔点的抽查。答不出来的，先去翻 `错题集.md` 对应条目。

**D-1（Day 7 周测）**
1. `sorted(big, key=..., reverse=True)` 之后，`big` 自己变了吗？
答：sorted不改变big，而是返回一个排序后的新列表
2. `[k for k, v in totals.items() if v > 40000]` 这个列表里的元素是什么类型？
元素类型是字符串
**D-3（Day 5 容器）**
3. `b = a`（a 是列表）之后改 `b`，`a` 会怎样？`a.copy()` 呢？
a会随b变化；a.copy()不随b变化。
**D-7（Day 1）**
4. 你写的第一份代码里，`if __name__ == "__main__":` 防的是什么？
只有直接运行的时候被执行，被import的时候不执行，防止被误导入，导致无限循环bug程序崩溃
   —— **今天它会第三次出现，而且你终于能完整理解它了。**

---
## 1. 为什么要把逻辑写成函数　*约 5 分钟*

不是为了"代码整洁"这种虚的理由。函数解决三个具体问题，每一个都很实际：

In [ ]:
# ── 理由一：可复用（同样的逻辑不写第二遍）──
def celsius_to_fahrenheit(c):   #摄氏度变为华氏度
    return c * 9 / 5 + 32

for c in [0, 100, 37]:
    print(f"  {c}°C = {celsius_to_fahrenheit(c)}°F")
print("  ↑ 换算规则只写了一次。要是没函数，你得复制三遍，改的时候漏一处就出错。")

print()
# ── 理由二：可测试（能单独喂进去一个值，看它吐什么）──
print("  assert celsius_to_fahrenheit(0) == 32, '0度应为32'")
assert celsius_to_fahrenheit(0) == 32
assert celsius_to_fahrenheit(100) == 212
print("  ✓ 两条断言通过 —— 有函数才能这样单独测它。一坨平铺代码没法单独测中间那段。")
print("     这就是 Day 6 那个 run_self_checks() 里能写 12 条 assert 的原因。")

print()
# ── 理由三：可读（给一段逻辑起个名字，读的人不用想它在干嘛）──
def is_valid_age(age):
    return 0 < age < 150

print(f"  is_valid_age(30) → {is_valid_age(30)}")
print("  ↑ 读主流程时看到 is_valid_age 就知道意图，不用琢磨 0 < age < 150 是干什么。")
print("     对比一下：如果 100 行代码里散落着 5 处 0 < age < 150，你读每一处都要重想一遍。")

  0°C = 32.0°F
  100°C = 212.0°F
  37°C = 98.6°F
  ↑ 换算规则只写了一次。要是没函数，你得复制三遍，改的时候漏一处就出错。

  assert celsius_to_fahrenheit(0) == 32, '0度应为32'
  ✓ 两条断言通过 —— 有函数才能这样单独测它。一坨平铺代码没法单独测中间那段。
     这就是 Day 6 那个 run_self_checks() 里能写 12 条 assert 的原因。

  is_valid_age(30) → True
  ↑ 读主流程时看到 is_valid_age 就知道意图，不用琢磨 0 < age < 150 是干什么。
     对比一下：如果 100 行代码里散落着 5 处 0 < age < 150，你读每一处都要重想一遍。


**这三条对你尤其重要，因为 vibe coding 的现实是：AI 一次给你 300 行平铺代码，你根本读不完。**
但如果它给你 12 个各 10 行的函数、每个有名字，你可以**只读名字和 docstring 就知道整体在干什么**。
**函数是把「一大坨不可验收的代码」变成「一堆可分别验收的小块」的手段。**

这也是为什么以后要求 AI 写代码时，你应该加一句：**「请拆成函数，别写成一整段平铺」**。

> ### ⏸ 检查点 1
> 能说清三个理由，尤其第三条（可验收性）。

---
## 2. `def` / `return` / 没有 return 会怎样　*约 7 分钟*

形状固定长这样：

```
def 函数名(参数1, 参数2):
    """一句话说明这个函数干什么"""      ← docstring
    做点什么
    return 结果                          ← 把值交出去
```

In [4]:
def add(a, b):
    return a + b

result = add(3, 4)          # 调用，把返回值接住
print("add(3, 4) =", result)

print()
# ── 关键：没有 return 的函数，返回的是 None ──
def greet(name):
    print(f"  你好，{name}")     # 只打印，不 return

x = greet("青承")
print("  greet 的返回值是:", repr(x))          # None
print(f"  类型: {type(x).__name__}")
print()
print("  ⚠️ 这是新手第二大坑（第一是 print 忘了逗号）：")
print("     函数【打印了东西】≠ 函数【返回了东西】。")
print("     你 print 出来的东西接不住，也传不下去。")

print()
# ── return 之后的代码永远不执行 ──
def check(n):
    if n < 0:
        return "负数，提前收工"
        print("这行永远不会执行")     # 死代码
    return "正数"

print(f"  check(-5) → {check(-5)}")
print("  ↑ return 一执行，函数立刻结束，后面的行直接作废。")
print("     这就是 Day 6 那个『先挡异常、再走正路』（提前收工）的机制。")

add(3, 4) = 7

  你好，青承
  greet 的返回值是: None
  类型: NoneType

  ⚠️ 这是新手第二大坑（第一是 print 忘了逗号）：
     函数【打印了东西】≠ 函数【返回了东西】。
     你 print 出来的东西接不住，也传不下去。

  check(-5) → 负数，提前收工
  ↑ return 一执行，函数立刻结束，后面的行直接作废。
     这就是 Day 6 那个『先挡异常、再走正路』（提前收工）的机制。


**🔮 预测**再运行：

In [ ]:
def f(a, b):
    a + b          # 注意：没有 return

print(f(1, 2))              # 你猜：None 没有return接不住返回值
print(repr(f(1, 2)))        # 你猜：None
print(f(f(1, 2), 3) if False else "跳过：因为 f(1,2) 是 None，None+3 会报错")

None
None
跳过：因为 f(1,2) 是 None，None+3 会报错


> ### ⏸ 检查点 2
> 必须刻住的三条：① 没 return 就是 None；② 打印≠返回；③ return 后面的代码是死代码。

---
## 3. 参数四件套　*约 10 分钟*

### 3.1 位置参数与关键字参数

In [6]:
def order_info(customer, item, qty):
    return f"{customer} 买了 {qty} 件 {item}"

# 位置参数：按顺序对号入座
print(order_info("青承社", "报表", 3))
# 关键字参数：写清 名字=值，顺序随便（可读性更好，长参数列表必用）
print(order_info(qty=5, item="看板", customer="福莱特"))
print()
print("  规则：一旦用了关键字参数，后面【必须全用它】，否则 SyntaxError。")
print("  实践建议：参数超过 2 个，就全部用关键字写法。读的人（和 AI）都省力。")

青承社 买了 3 件 报表
福莱特 买了 5 件 看板

  规则：一旦用了关键字参数，后面【必须全用它】，否则 SyntaxError。
  实践建议：参数超过 2 个，就全部用关键字写法。读的人（和 AI）都省力。


### 3.2 默认值——最有用也最容易踩

**你已经用过它了**：`sorted(big, key=..., reverse=True)` 里你没写 key 的话默认是 `None`；
`enumerate(fruits, start=1)` 里 `start` 默认是 0。默认值让调用方能少写东西。

In [7]:
def query(table, limit=100, desc=False):
    order = "降序" if desc else "升序"
    return f"查 {table}，取 {limit} 条，{order}"

print(query("orders"))                    # 全用默认
print(query("orders", limit=50))          # 只改一个
print(query("orders", 20, True))          # 位置写法（能用但可读性差）
print()

# ⚠️ 经典大坑：默认值不能用可变对象（列表/字典）
def bad_append(item, bucket=[]):        # 看起来每次调用都是新空列表？
    bucket.append(item)
    return bucket

print("  bad_append(1) →", bad_append(1))
print("  bad_append(2) →", bad_append(2), " ← 上次的 1 还在！！")
print("  bad_append(3) →", bad_append(3))
print()
print("  原因：默认值【只在 def 那一刻创建一次】，之后所有调用共用同一个列表。")
print("  正确写法：默认 None，函数内部再造。")

def good_append(item, bucket=None):
    if bucket is None:                  # 判断 None 用 is，Day 3 讲的
        bucket = []
    bucket.append(item)
    return bucket

print("  good_append(1) →", good_append(1))
print("  good_append(2) →", good_append(2), " ✓ 干净")

查 orders，取 100 条，升序
查 orders，取 50 条，升序
查 orders，取 20 条，降序

  bad_append(1) → [1]
  bad_append(2) → [1, 2]  ← 上次的 1 还在！！
  bad_append(3) → [1, 2, 3]

  原因：默认值【只在 def 那一刻创建一次】，之后所有调用共用同一个列表。
  正确写法：默认 None，函数内部再造。
  good_append(1) → [1]
  good_append(2) → [2]  ✓ 干净


**🔮 预测**：下面这个会输出什么？先猜。

In [ ]:
def counter(n, acc=[]):
    acc.append(n)
    return acc

print(counter(1))     # 你猜：[1]
print(counter(2))     # 你猜：[1, 2]
print(counter(3))     # 你猜：[1, 2, 3]

### 3.3 `*args` 与 `**kwargs`——**今天只要求读懂，不要求会写**

你会在 AI 生成的代码和库里大量看到它们。含义是"接住剩下的"：
- `*args` → 把多余的位置参数收进一个**元组**
- `**kwargs` → 把多余的关键字参数收进一个**字典**

In [ ]:
def show_all(*args, **kwargs):
    print("  args   （位置参数们）→", args, f"  类型 {type(args).__name__}")
    print("  kwargs（关键字参数们）→", kwargs, f"  类型 {type(kwargs).__name__}")

show_all(1, 2, 3, name="青承", city="上海")
print()
print("  最常见的真实用途：写一个『转发参数』的包装函数，")
print("  以及库函数签名里的 def foo(*args, **kwargs) 表示『我接受任意参数』。")
print()

# 反向用法（这个你以后一定会遇到）：把列表/字典『拆开』传进去
def make_order(customer, item, qty):
    return f"{customer}/{item}/{qty}"

order_data = ("福莱特", "光伏玻璃", 12000)
print("  用 * 拆开元组当位置参数:", make_order(*order_data))

order_dict = {"item": "胶膜", "customer": "青承社", "qty": 300}
print("  用 ** 拆开字典当关键字参数:", make_order(**order_dict))
print()
print("  ↑ 这个 * 和 ** 的『拆开』用法，比它的『收集』用法更常出现在你的代码里。")

> ### ⏸ 检查点 3
> 今天必留下的两条：**① 默认值不能写可变对象（用 None 再判）；② `*`/`**` 还能反过来用——把容器拆成参数。**

---
## 4. 返回多个值（元组解包）　*约 5 分钟*

**这个你 Day 5 就见过**：`for k, v in d.items()` 里的 `k, v` 就是元组解包。
函数也能一次给你多个值——本质是返回一个元组，调用方拆开接。

In [ ]:
def parse_amount(text):
    """返回：数值 和 币种。"""
    cleaned = text.strip().replace(",", "")
    currency = "USD" if cleaned.startswith("$") else "CNY"
    value = float(cleaned.replace("$", ""))
    return value, currency          # 逗号 = 打包成元组

result = parse_amount("$ 1,250.50")
print("  接到的是一个元组:", result, f"  类型 {type(result).__name__}")
print()

# 拆开接（这就是元组解包）
value, currency = parse_amount("$ 1,250.50")
print(f"  value   = {value}")
print(f"  currency = {currency}")
print()

# ⚠️ 拆的数量必须对上
try:
    a, b, c = parse_amount("$10")
except ValueError as e:
    print(f"  ValueError: {e}")
print()
print("  实践建议：返回值超过 3 个就别用元组了，改回【字典】——")
print("  调用方按键取值，不用记顺序。你的清洗工具里 r 就是字典，同一个道理。")

> ### ⏸ 检查点 4

---
## 5. ⚠️ 作用域与全局变量　*约 10 分钟 —— 本周最重要的一节*

**这节和第 5 天「赋值不是复制」是同一个家族：都是「隐蔽的共享状态」。**
你第 1 周栽过的最难查的坑基本都属这一类。

### 5.1 函数内部创建的变量，外面看不见

In [ ]:
def make():
    inner = 42            # 局部变量，函数结束就销毁
    return inner

make()
try:
    print(inner)
except NameError as e:
    print(f"  NameError: {e}")
print("  ↑ inner 只活在函数运行期间。这是好事 —— 它不会污染外面。")

### 5.2 但外面的变量，函数里**能读**——这就是坑的开始

In [ ]:
total = 0                    # 全局变量

def add_up(items):
    s = 0
    for i in items:
        s += i
    return s

print("  add_up([1,2,3]) =", add_up([1, 2, 3]))
print("  函数里没定义 total，但能读到它 ↓")

def bad_show():
    print("  函数内部读到了全局的 total =", total)     # 能读，不报错

bad_show()

### 5.3 想**改**全局变量，必须写 `global`——而这几乎总是设计错误

In [ ]:
count = 0

def bump_wrong():
    count = count + 1        # 看起来像 +1，实际是【新建了一个同名局部变量】

try:
    bump_wrong()
except UnboundLocalError as e:
    print(f"  UnboundLocalError: {e}")
print("  ↑ 因为赋值让 Python 认定 count 是局部的，于是右边那个 count 变成『读未赋值的局部变量』。")
print("     这个报错非常反直觉，新手会盯着这行看十分钟觉得它没问题。")
print()

def bump_right():
    global count             # 能这么写，但别这么用
    count += 1
    return count

print("  bump_right() →", bump_right())
print("  bump_right() →", bump_right())
print("  现在全局 count =", count, " ← 函数偷偷改了外面的东西")

### 5.4 最隐蔽的一种：改全局**可变对象**，连 `global` 都不用写

In [ ]:
log = []                          # 全局列表

def record(msg):
    log.append(msg)               # ⚠️ 没写 global，但确实改了全局的 log

record("启动")
record("读完文件")
print("  log =", log)
print()
print("  ↑ 因为 append 没有给 log 赋值，它是在【改那个对象本身】。")
print("     所以 Python 允许你改，不会报错，也不会有任何提示。")
print()
print("  【这才是真正危险的地方】：函数看起来是个干净的纯计算，")
print("  实际上偷偷往全局里塞东西。调用顺序一变、或者多线程一跑，")
print("  就会出现『同一份输入，两次跑出不同结果』这种要命的问题。")
print("  和第 5 天那个『改副本的字典，原件一起变』是同一族病。")

### 5.5 结论：写函数时守一条规矩

> **函数要像一台小机器：东西从参数进去，结果从 return 出来，中间不碰外面的世界。**

这在专业上叫「纯函数」。好处极其实际：
- **能单独测**（不用先构造全局状态）
- **能随便调顺序**（不依赖调用历史）
- **AI 更容易写对**（上下文少 = 出错少）
- **出了问题只怪这一个函数**（不用在全文件里追哪个地方改过变量）

In [ ]:
# ── 坏写法：靠全局状态串起来 ──
_state = {}
def bad_step_a(data):
    _state["clean"] = [d.strip() for d in data]
def bad_step_b():
    _state["total"] = len(_state["clean"])        # 必须先调 a 才能调 b！
print("  坏写法：b 依赖 a 的副作用，调用顺序错了就 KeyError，")
print("           而且函数签名里完全看不出这个依赖关系。")
print()

# ── 好写法：数据显式传来传去 ──
def clean(data):
    return [d.strip() for d in data]

def count_total(items):
    return len(items)

raw = ["  a ", "b"]
step1 = clean(raw)
step2 = count_total(step1)
print(f"  好写法：{raw} → {step1} → 共 {step2} 条")
print("  ↑ 每个函数需要什么，看参数就知道。谁先谁后一目了然。")
print()
print("  你的 day06_cleaner.py 已经是好写法的雏形了 ——")
print("  第1步产 good_rows，第2步吃 cleaned，第3步吃 usable。数据是往下传的，")
print("  不是靠全局变量互相偷看。今天练习 A 就是把它彻底函数化。")

**LEGB——名字查找的顺序，记这四个字母就够了**：

| 层 | 英文 | 是什么 |
|---|---|---|
| L | Local | 函数内部 |
| E | Enclosing | 外层函数（闭包，今天不学）|
| G | Global | 模块顶层 |
| B | Built-in | Python 自带的（`list`、`print`、`sum`...）|

**Python 由内向外一层层找，找到就用。** 这解释了两件你已经遇到的事：
- Day 5 你写 `list = [1,2,3]` 之后 `list([4,5])` 崩了 → 你把 **L/G 层的 list 顶掉了 B 层的**
- 今天 `count = count + 1` 报 UnboundLocalError → 赋值让 count 提前出现在 **L 层**，于是右边读的是还没赋值的 L 层

**所以：变量名永远别用 `list` `dict` `str` `type` `sum` `max` `id` `input` `format`。**

> ### ⏸ 检查点 5（重点）
> 这一节的判断标准：你能不能对任何一个函数回答——
> **「它读了外面的什么东西？它改了外面的什么东西？」**
> 答案是「都没有，全靠参数和返回值」，这个函数就是干净的。

---
## 6. docstring 与类型标注　*约 6 分钟 —— 对 AI 有直接实效*

### 6.1 docstring：函数下面那行三引号

In [ ]:
def summarize_amounts(rows, key="金额"):
    """按指定键汇总金额。

    参数:
        rows: 字典组成的列表，例如 [{"金额": 100}, {"金额": 50}]
        key:  要汇总的字段名，默认 "金额"

    返回:
        float，所有行的金额总和。缺失的字段按 0 计算。

    用法:
        summarize_amounts([{"金额": 100}, {"金额": 50}])  → 150.0
    """
    total = 0.0
    for r in rows:
        v = r.get(key)
        if v is not None:
            total += v
    return total

print(summarize_amounts([{"金额": 100}, {"金额": 50}]))
print()
print("  ↑ 这个 docstring 不是写给人看的装饰，它有三个实际用途：")
print("     ① 别人（未来的你）不用读代码就知道边界行为（缺失按0、返回float）")
print("     ② IDE 鼠标悬停会显示它")
print("     ③ 你把这段贴给我，我立刻能准确用这个函数，不用猜")

### 6.2 类型标注：为什么它对 AI 有实效

语法是 `参数: 类型` 和 `-> 返回类型`。**Python 不做强制检查**（标错了也能跑），
它纯粹是给人和 AI 的说明书。

In [ ]:
def calc_bmi_typed(height_m: float, weight_kg: float) -> float:
    return weight_kg / (height_m ** 2)

print(f"  calc_bmi_typed(1.75, 70) = {calc_bmi_typed(1.75, 70):.2f}")
print()

def find_max_amounts(rows: list[dict], threshold: float = 0) -> list[str]:
    result = []
    for r in rows:
        if r["金额"] > threshold:
            result.append(r["品名"])
    return result

demo = [{"品名": "玻璃", "金额": 9000}, {"品名": "胶膜", "金额": 100}]
print(f"  find_max_amounts(demo) = {find_max_amounts(demo)}")
print(f"  加阈值 threshold=5000 → {find_max_amounts(demo, threshold=5000)}")
print()
print("  读一下第二行签名: def find_max_amounts(rows: list[dict], threshold: float = 0) -> list[str]")
print("    rows 要传一个『装了字典的列表』")
print("    threshold 是浮点数，可以不传（默认 0）")
print("    返回的是『装了字符串的列表』")
print()
print("  ★ 对 AI 的实际价值：")
print("    没有标注时，你说『传数据进去』，AI 得猜是字典还是列表、单个还是多个；")
print("    有标注时，类型就是精确契约，AI 生成的调用代码几乎不会错。")
print("    这就是课程表里那句『有类型标注的代码 AI 写得更准』的技术含义。")
print()
print("  ⚠️ 但要知道：类型标错了 Python 不会拦你 ↓")
print(f"     calc_bmi_typed('1.75', 70) 会报错，但报错发生在运行时的除法，")
print("     而不是调用那一行的类型检查。所以标注是【给读者的承诺】，不是保险。")
try:
    calc_bmi_typed("1.75", 70)
except TypeError as e:
    print(f"     实测: TypeError: {e}")

**今天记住两个动作就行**：① 每个函数写一行 docstring 说清『缺失时怎么办』；
② 参数和返回值加类型标注。这两件事让你的代码从"能跑"变成"能被别人正确调用"。

> ### ⏸ 检查点 6

---
## 7. class：今天**只要求读懂**　*约 8 分钟*

**明确预期**：课程表写的是"读懂即可，不要求会设计"。你今天只需要能认出这些东西：

- `class 名字:` 定义一个类（一种自定义的数据类型）
- `def __init__(self, ...)` 是"造这个东西时的初始化"
- `self` 是"这个对象自己"，通过它访问自己的数据
- `def 方法(self, ...)` 是这个对象能做的动作

In [ ]:
class Order:
    def __init__(self, customer: str, item: str, qty: int, price: float):
        # 把传进来的值，存到这个对象自己的属性上
        self.customer = customer
        self.item = item
        self.qty = qty
        self.price = price

    def amount(self) -> float:
        # 一个动作：数量乘单价
        return self.qty * self.price

    def describe(self) -> str:
        return f"{self.customer} 的 {self.item}，{self.qty} 件，共 {self.amount():,.0f}"


# ── 用：造对象，然后调它的方法 ──
o1 = Order("福莱特", "光伏玻璃", 12000, 85.5)
o2 = Order("青承社", "胶膜", 300, 1200.0)

print(f"  o1.amount() = {o1.amount():,.0f}")
print(f"  o1.describe() = {o1.describe()}")
print()
orders = [o1, o2]                       # 对象可以装进列表
print(f"  列表里的对象合计: {sum(o.amount() for o in orders):,.0f}")
print()
print("  读法要点：")
print("    o1.amount()  ← 传参只有 self，所以调用时【不用传任何东西】")
print("    self.qty 里的 self 就是 o1 自己，它读的是 o1.qty")
print()
print("  ★ 现在你终于能完整理解 class 和函数的分工了：")
print("     函数 = 一个动作。class = 一份数据 + 能对这份数据做的那些动作，绑在一起。")
print("     Day 6 的 rows 是纯数据、amount_of() 是纯函数；")
print("     Order 把它俩绑成了一个东西，传参时不用再说『这是数据，那是规则』。")

**你现在不需要会写 class。** 但你要能在 AI 给你的代码里认出它，
并且知道：看到 `class` 时，先看 `__init__` 有哪几个 `self.xxx`（那就是这个对象携带的数据），
再看有哪些方法（那就是它能做的事）。**读 class 就这两个动作。**

> ### ⏸ 检查点 7

---
# 🔨 今日练习

**练习 A 是今天的核心产出，也是课程表点名的：重构你自己的代码。**

## 练习 A：把 day06_cleaner.py 彻底函数化　*必做，约 15 分钟*

你那个工具的第 1、2 步现在是**两大段平铺代码**——能跑，但不好测、不好复用、不好改。

**任务**：把 `week01/day06_cleaner.py` 重构成下面这个形状（我建议的函数划分）：

```python
def split_lines(path: str) -> list[str]:
    """读文件，返回每一行的文本。"""

def classify_lines(lines, header, expected_fields):
    """把行分成 可用/噪声/修补/坏行 四类。返回四个值（元组解包）。"""

def clean_row(parts: list[str]) -> dict:
    """把一行拆好的字段清洗成一条规范记录字典。"""

def validate_and_recompute(records):
    """业务校验 + 金额重算。返回 可用行/业务坏行/金额不符 三部分。"""

def summarize_by_category(usable: list[dict]) -> tuple[dict, float]:
    """按类别汇总。返回 (统计字典, 总额)。"""

def render_report(...) -> None:
    """打印汇总表和五类报告。只做事，不返回。"""

if __name__ == "__main__":
    # 主流程变成 6-8 行，读起来像一句话一句话
```

**重构的验收标准**：
1. 跑出来的输出**必须和原来逐字一致**（13 行汇总、4,131,260.00、五类报告计数）
2. 12 条 assert 全部保留并通过
3. 没有任何函数读写全局变量——**数据只从参数进、从 return 出**（第 5 节的规矩）
4. 每个函数有 docstring + 至少参数有类型标注
5. 主流程短到你能一眼讲清整个程序做了什么

⚠️ **这是你第一次做重构。重构的定义是:改结构,不改行为。** 所以第 1 条是硬标准。
做法建议:先复制一份成 `day06_cleaner_v2.py`,改完两份都跑,`diff` 对比输出。
**别在唯一一份上直接改**——万一改坏了你没有对照物。这就是 Git 存在的原因(明天 Day 9 讲)。

**这一题我不会给你完整代码**,因为重构本身就是本周要练的能力。但我可以：
- 帮你写第一个函数 `split_lines` 作为样板
- 你写完每一个函数我帮你 review
- 卡住 15 分钟就来问,别自己耗

## 练习 B：给 bmi_calculator.py 补类型标注　*必做，约 5 分钟*

打开 `week01/bmi_calculator.py`,把三个函数的签名改成带类型标注的:

```python
def calc_bmi(height_m: float, weight_kg: float) -> float | None:
def bmi_category(bmi_value: float) -> str:
def run_self_checks() -> int:
```

`float | None` 的意思是"可能是 float,也可能是 None"——**这正好是它实际的行为**
(身高传 0 时它返回 None)。这个写法叫联合类型,Python 3.10+ 支持,读 AI 代码时极常见。

改完运行一次确认没弄坏。**然后回答**:为什么标注 `-> float` 是错的,而 `-> float | None` 才对?

## 练习 C：读懂一段 class　*选做，约 5 分钟*

把下面这段跑起来,然后回答:它一共有几个方法?`self.name` 这个数据是在哪一行被创建出来的?

In [ ]:
class Student:
    def __init__(self, name, days_done):
        self.name = name
        self.days_done = days_done

    def progress(self):
        return self.days_done / 84

    def report(self):
        return f"{self.name}: Day {self.days_done}，进度 {self.progress():.1%}"

s = Student("kisaragi", 8)
print("  " + s.report())
print(f"  再加 3 天: ", end="")
s.days_done += 3
print(s.report())

---
## 今日收尾

1. **NOTES.md 加 Day 8 三行笔记**。这次多加一栏(第 1 周立的机制):
   **「今天有什么我以为对了、但其实错了？」** → 答不出的话,至少写「今天验证了什么」。
2. **错题集**:今天新的两条(默认值用可变对象、全局可变对象被偷改)先标 ⬜,周日重做。
3. 提交:
```
cd ~/Documents/AI自学/ai-product-journey
git add .
git status
git commit -m "Day 8: 函数与代码组织 + 清洗器重构"
git push
```

明天 Day 9 是 **文件读写、JSON/CSV、异常处理**——正好接今天:你的重构里
`split_lines` 会把「读文件」单独包出来,那时会讲为什么要把这种「会失败的操作」隔离开。